In [2]:
# ========================
# PARTE 0: INSTALAÇÕES (Google Colab)
# ========================
!apt-get update > /dev/null
!apt install chromium-chromedriver -y > /dev/null || true
!cp /usr/lib/chromium-browser/chromedriver /usr/bin || true
!pip install selenium > /dev/null
!pip install pandas openpyxl > /dev/null

# ========================
# PARTE 1: IMPORTS E CONFIGURAÇÕES
# ========================
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import NoSuchElementException, TimeoutException, StaleElementReferenceException
import pandas as pd
import os, time, zipfile, requests
from datetime import datetime

options = Options()
options.binary_location = "/usr/bin/chromium-browser"
options.add_argument("--headless")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
driver = webdriver.Chrome(options=options)
wait = WebDriverWait(driver, 15)

# ========================
# PARTE 2: COLETA DE LINKS VIA API ALGOLIA
# ========================
ALGOLIA_APP_ID = "1CF3ZT43ZU"
ALGOLIA_API_KEY = "150c68d1c61fc1835826a57a203dab72"
ALGOLIA_INDEX = "production_products"

headers_algolia = {
    "Content-Type": "application/json",
    "X-Algolia-Application-Id": ALGOLIA_APP_ID,
    "X-Algolia-API-Key": ALGOLIA_API_KEY,
}

def buscar_links_algolia(query="revestimento", max_paginas=30, limite=10):
    links = set()
    for pagina in range(max_paginas):
        url = f"https://{ALGOLIA_APP_ID}-dsn.algolia.net/1/indexes/{ALGOLIA_INDEX}/query"
        payload = {
            "params": f"query={query}&page={pagina}&hitsPerPage=100"
        }
        res = requests.post(url, headers=headers_algolia, json=payload)
        if res.status_code != 200:
            break
        data = res.json()
        hits = data.get("hits", [])
        if not hits:
            break
        for item in hits:
            relative_url = item.get("url")
            if relative_url:
                if relative_url.startswith("http"):
                    full_url = relative_url
                else:
                    full_url = f"https://www.leroymerlin.com.br{relative_url}"
                if "/revestimento" in full_url:
                    links.add(full_url)
                    if len(links) >= limite:
                        return links
        if pagina >= data.get("nbPages", 0) - 1:
            break
    return links

# Coletar todos os links via Algolia
print("\U0001F50D Coletando links via API Algolia...")
todos_links = buscar_links_algolia(query="revestimento", limite=10)
print(f"✅ Total de links coletados via API: {len(todos_links)}\n")

# ========================
# PARTE 3: EXTRAÇÃO DE DADOS
# ========================
def extrair_dados_produto(url):
    driver.get(url)
    try:
        wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "h1")))
    except TimeoutException:
        return {'URL': url, 'Erro': 'Timeout'}

    dados = {'URL': url}
    dados['Nome'] = driver.find_element(By.CSS_SELECTOR, "h1").text if driver.find_elements(By.CSS_SELECTOR, "h1") else None
    dados['Código'] = url.split("_")[-1] if "_" in url else None

    precos_candidatos = [
        '[data-cy="product-price"]',
        '[data-cy="product-price-integer"]',
        '[data-cy="product-price-sale"]',
        '.price__sales',
        '.value__integer'
    ]
    preco_txt = None
    for sel in precos_candidatos:
        elems = driver.find_elements(By.CSS_SELECTOR, sel)
        if elems:
            text = elems[0].text.strip()
            if text:
                preco_txt = text
                break

    if preco_txt:
        clean = preco_txt.replace("R$", "").replace("/m²", "").replace(".", "").replace(",", ".").strip()
        try:
            preco_val = float(clean)
            dados['Preço Promocional'] = f"{preco_val:.2f}".replace(".", ",")
        except:
            dados['Preço Promocional'] = "N/A"
    else:
        dados['Preço Promocional'] = "N/A"

    dados['Descrição'] = driver.find_element(By.CSS_SELECTOR, 'meta[name="description"]').get_attribute("content") if driver.find_elements(By.CSS_SELECTOR, 'meta[name="description"]') else None
    dados['Imagem'] = driver.find_element(By.CSS_SELECTOR, 'meta[property="og:image"]').get_attribute("content") if driver.find_elements(By.CSS_SELECTOR, 'meta[property="og:image"]') else "N/A"

    specs = {}
    for tr in driver.find_elements(By.CSS_SELECTOR, 'table tr'):
        try:
            k = tr.find_element(By.CSS_SELECTOR, 'th').text.strip()
            v = tr.find_element(By.CSS_SELECTOR, 'td').text.strip()
            specs[k] = v
        except:
            continue
    dados['Especificações'] = "\n".join(f"{k}: {v}" for k, v in specs.items())
    dados['Loja'] = "Leroy Merlim"

    return dados

# ========================
# PARTE 4: COLETAR TODOS OS PRODUTOS
# ========================
todos = []
for i, link in enumerate(sorted(todos_links), 1):
    print(f"\U0001F50D {i}/{len(todos_links)} -> {link}")
    try:
        todos.append(extrair_dados_produto(link))
    except Exception as err:
        print("Erro:", err)
driver.quit()

# ========================
# PARTE 5: SALVAR PLANILHA E CSV COM ORDEM DEFINIDA
# ========================
data_nome = datetime.now().strftime("%Y-%m-%d")
pasta_base = "leroymerelin_coleta"
os.makedirs(pasta_base, exist_ok=True)

arquivo_excel = f"{pasta_base}/leroymerelin_{data_nome}.xlsx"
arquivo_csv = f"{pasta_base}/leroymerelin_{data_nome}.csv"

colunas_ordenadas = ['Código', 'Nome', 'Preço Promocional', 'Descrição', 'Especificações', 'Imagem', 'URL', 'Loja']

if not todos:
    print("❌ Nenhum produto foi coletado. Verifique os links ou a estrutura da página.")
else:
    df = pd.DataFrame(todos)
    df.drop(columns=[c for c in df.columns if c in ['Erro']], inplace=True, errors='ignore')
    df = df[colunas_ordenadas]
    df.to_excel(arquivo_excel, index=False)
    df.to_csv(arquivo_csv, index=False, sep=";")

    print(f"\U0001F4C4 Excel salvo em: {arquivo_excel}")
    print(f"\U0001F4C4 CSV salvo em: {arquivo_csv}")

    # ========================
    # PARTE 6: BAIXAR IMAGENS E ZIPAR
    # ========================
    pasta_imagens = os.path.join(pasta_base, f"imagens_leroymerelin_{data_nome}")
    os.makedirs(pasta_imagens, exist_ok=True)

    for _, r in df.iterrows():
        url_img = r['Imagem']
        cod = r['Código']
        if url_img and url_img != "N/A":
            for t in range(3):
                try:
                    resp = requests.get(url_img, headers={"User-Agent":"Mozilla/5.0"}, timeout=10, stream=True)
                    if resp.status_code == 200:
                        with open(f"{pasta_imagens}/{cod}.png", "wb") as f:
                            for cb in resp.iter_content(1024): f.write(cb)
                        break
                except:
                    time.sleep(2)

    zipname = os.path.join(pasta_base, f"imagens_leroymerelin_{data_nome}.zip")
    with zipfile.ZipFile(zipname, "w", zipfile.ZIP_DEFLATED) as z:
        for fn in os.listdir(pasta_imagens):
            z.write(os.path.join(pasta_imagens, fn), fn)
    print(f"\U0001F4E6 Imagens zipadas em: {zipname}")


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


cp: '/usr/lib/chromium-browser/chromedriver' and '/usr/bin/chromedriver' are the same file
🔍 Coletando links via API Algolia...
✅ Total de links coletados via API: 10

🔍 1/10 -> https://www.leroymerlin.com.br/revestimento-acetinado-32x55cm-32700-cristofoletti_92358322
🔍 2/10 -> https://www.leroymerlin.com.br/revestimento-decorativo-portugues-15,4x15,4cm-coimbra-imper-sofistiq_92261603
🔍 3/10 -> https://www.leroymerlin.com.br/revestimento-interno-acetinado-borda-arredondada-33x60cm-canjiquinha-white-hd53-1096-embramaco_90779031
🔍 4/10 -> https://www.leroymerlin.com.br/revestimento-interno-amadeirado-acetinado-borda-reta-32x60cm-rig-legno-biancogres_90543691
🔍 5/10 -> https://www.leroymerlin.com.br/revestimento-interno-brilhante-borda-arredondada-32,3x57,3cm-prisma-colorido-ceral_91082075
🔍 6/10 -> h